# Step 1a: Spatial Data Download

This notebook downloads the required data files for preprocessing.

## Libraries

In [1]:
import requests
import zipfile

import pandas as pd

from datetime import datetime, timedelta
from IPython.display import display, HTML
from pathlib import Path
from time import time

## Functions

In [2]:
# Downloads a file from a link
def download(url, file):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(file, 'wb') as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

# Unzips a single compressed .zip file
def unzip(file, folder):
    ext = Path(file).suffixes[0].lower()
    with zipfile.ZipFile(file, 'r') as z:
        if ext == '.shp':
            for target in z.infolist():
                zext = ''.join(Path(target.filename).suffixes).replace(ext, '')
                target.filename = Path(file).stem + zext
                z.extract(target, folder)
        else:
            target = z.infolist()[0]
            target.filename = Path(file).stem
            z.extract(target, folder)
        
# Formats human readable sizes in bytes
# https://stackoverflow.com/questions/1094841/get-human-readable-version-of-file-size
def format_size(num, suffix="B"):
    for unit in ["", "Ki", "Mi", "Gi", "Ti", "Pi", "Ei", "Zi"]:
        if abs(num) < 1024.0:
            return f"{num:3.1f} {unit}{suffix}"
        num /= 1024.0
    return f"{num:.1f} Yi{suffix}"

# Extracts file info for summary
def file_info(file):
    stats = Path(file).stat()
    modified = datetime.fromtimestamp(stats.st_mtime)
    created = datetime.fromtimestamp(stats.st_birthtime)
    size = format_size(stats.st_size)
    out = {'created': created, 'modified': modified, 'size': size}
    return out

## Settings

In [3]:
folder = '../../tmp/downloads'
data_sources = pd.read_csv('data.csv')

## Run

Run the download process for all defined data files in the settings.

In [4]:
# Create folder if not exists
Path(folder).mkdir(parents=True, exist_ok=True)

### Download Data

Download data from url sources to ``folder``.

In [5]:
# Start download
print(f'Starting Downloads ({datetime.now()})...')
start = time()
for row in data_sources.itertuples():
    
    # Data file vars
    url = row.url
    file = f'{folder}/{row.file}'
    
    # Download data file if it does not exist otherwise skip
    if not Path(file).is_file():
        print(f'Downloading {row.file} ({datetime.now()})...')
        download(url, file)
        print(f'Downloaded {row.file} ({datetime.now()})')
    else:
        print(f'Skipping {row.file} - file exists ({datetime.now()})...')
        
# End downloads
end = time()
elapsed = str(timedelta(seconds=end - start))
print(f'Downloads Complete ({datetime.now()})')
print(f'Elapsed Time ({elapsed})')

Starting Downloads (2025-01-20 19:15:04.807539)...
Skipping toronto.geojson - file exists (2025-01-20 19:15:04.808328)...
Skipping centrelines.geojson - file exists (2025-01-20 19:15:04.808444)...
Downloaded collisions.geojson (2025-01-20 19:15:37.978780)
Downloaded traffic.csv (2025-01-20 19:15:43.558825)
Downloaded autospeed_enforcement.geojson (2025-01-20 19:15:43.753339)
Downloaded watch_your_speed.geojson (2025-01-20 19:15:43.997479)
Downloaded red_light_cams.geojson (2025-01-20 19:15:44.308833)
Downloaded police.geojson (2025-01-20 19:15:44.498941)
Downloaded ambulance.geojson (2025-01-20 19:15:44.714919)
Downloaded fire_hydrants.geojson (2025-01-20 19:15:46.411868)
Downloaded fire_stations.geojson (2025-01-20 19:15:46.670804)
Downloaded renewables.geojson (2025-01-20 19:15:46.936076)
Downloaded bicycle_parking.geojson (2025-01-20 19:15:47.815201)
Downloaded transit_shelters.geojson (2025-01-20 19:15:48.244318)
Downloaded wayfind.geojson (2025-01-20 19:15:48.466950)
Downloaded li

### Unzip Data

Unzip compressed data files.

In [6]:
# Unzip data files
print(f'Unzipping Data ({datetime.now()})...')
start = time()
for row in data_sources.itertuples():
    
    # Data file vars
    url = row.url
    file = f'{folder}/{row.file}'
    
    # Download data file if it does not exist otherwise skip
    path = Path(file)
    if path.suffix.lower() == '.zip':
        if path.with_suffix('').is_file():
            print(f'Skipping {row.file} - already unzipped ({datetime.now()})...')
        else:
            print(f'Unzipping {row.file}')
            unzip(file, folder)
            print(f'Unzipped {row.file} ({datetime.now()})')
    else:
        print(f'Skipping {row.file} - not a zip file ({datetime.now()})...') 

# End downloads
end = time()
elapsed = str(timedelta(seconds=end - start))
print(f'Unzip Complete ({datetime.now()})')
print(f'Elapsed Time ({elapsed})')

Unzipping Data (2025-01-20 19:16:15.472873)...
Skipping toronto.geojson - not a zip file (2025-01-20 19:16:15.473547)...
Skipping centrelines.geojson - not a zip file (2025-01-20 19:16:15.473571)...
Skipping collisions.geojson - not a zip file (2025-01-20 19:16:15.473587)...
Skipping traffic.csv - not a zip file (2025-01-20 19:16:15.473601)...
Skipping autospeed_enforcement.geojson - not a zip file (2025-01-20 19:16:15.473615)...
Skipping watch_your_speed.geojson - not a zip file (2025-01-20 19:16:15.473628)...
Skipping red_light_cams.geojson - not a zip file (2025-01-20 19:16:15.473642)...
Skipping police.geojson - not a zip file (2025-01-20 19:16:15.473655)...
Skipping ambulance.geojson - not a zip file (2025-01-20 19:16:15.473668)...
Skipping fire_hydrants.geojson - not a zip file (2025-01-20 19:16:15.473681)...
Skipping fire_stations.geojson - not a zip file (2025-01-20 19:16:15.473693)...
Skipping renewables.geojson - not a zip file (2025-01-20 19:16:15.473706)...
Skipping bicycle

## Summary

In [7]:
# Copy data sources to use in summary
summary = data_sources.copy()

# Get file infos and add to summary
info = [file_info(f'{folder}/{row.file}') for row in summary.itertuples()]
info = pd.DataFrame(info)
summary = pd.concat([summary, info], axis=1)

# Rearrange summary columns and display
summary = summary[['file', 'size', 'created', 'modified', 'source', 'source_url', 'url']]
display(HTML(summary.to_html(render_links=True)))

,file,size,created,modified,source,source_url,url
0,toronto.geojson,2.0 MiB,2025-01-20 19:09:19.728592,2025-01-20 19:09:19.890474,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/neighbourhoods/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/neighbourhoods/resource/1d38e8b7-65a8-4dd0-88b0-ad2ce938126e/download/Neighbourhoods%20-%204326.geojson
1,centrelines.geojson,89.1 MiB,2025-01-20 19:09:20.074805,2025-01-20 19:09:26.098334,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/toronto-centreline-tcl/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/1d079757-377b-4564-82df-eb5638583bfb/resource/7bc94ccf-7bcf-4a7d-88b1-bdfc8ec5aaf1/download/Centreline%20-%20Version%202%20-%204326.geojson
2,collisions.geojson,401.7 MiB,2025-01-20 19:15:05.265715,2025-01-20 19:15:37.962901,Toronto Police Service Public Safety Data Portal,https://data.torontopolice.on.ca/datasets/TorontoPS::traffic-collisions-open-data-asr-t-tbl-001/about,https://stg-arcgisazurecdataprod.az.arcgis.com/exportfiles-6104-20398/Traffic_Collisions_Open_Data_-6024052409346627848.geojson?sv=2018-03-28&sr=b&sig=T%2FtlvA7HFHIHAPCJuiKXMSsqhHvJ8472JrgX7vF8x0o%3D&se=2025-01-21T01%3A07%3A03Z&sp=r
3,traffic.csv,77.6 MiB,2025-01-20 19:15:38.206589,2025-01-20 19:15:43.546898,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/traffic-volumes-at-intersections-for-all-modes/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/traffic-volumes-at-intersections-for-all-modes/resource/1f60c668-bb8e-4e1e-ac72-3c6558a03fea/download/raw-data-2010-2019.csv
4,autospeed_enforcement.geojson,38.8 KiB,2025-01-20 19:15:43.749028,2025-01-20 19:15:43.753120,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/automated-speed-enforcement-locations/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/a154790c-4a8a-4d09-ab6b-535ddb646770/resource/9842895b-2b8b-4b60-9320-c0a1fde4afd8/download/Automated%20Speed%20Enforcement%20Locations%20-%204326.geojson
5,watch_your_speed.geojson,487.4 KiB,2025-01-20 19:15:43.946778,2025-01-20 19:15:43.995852,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/school-safety-zone-watch-your-speed-program-locations/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/642efeca-1258-4c23-8e86-d9affca26001/resource/b3268057-6b55-4e6e-b1ae-b752bce92a1c/download/Stationary%20Sign%20locations%20-%204326.geojson
6,red_light_cams.geojson,242.8 KiB,2025-01-20 19:15:44.249247,2025-01-20 19:15:44.307499,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/red-light-cameras/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/9fcff3e1-3737-43cf-b410-05acd615e27b/resource/7e4ac806-4e7a-49d3-81e1-7a14375c9025/download/Red%20Light%20Cameras%20Data%20-%204326.geojson
7,police.geojson,8.0 KiB,2025-01-20 19:15:44.497979,2025-01-20 19:15:44.498726,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/police-facility-locations/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/9aeefa17-27e8-4dd9-b74d-80f7f9eb85ac/resource/c0176e24-8b76-4bb2-96fa-61cc1af2a065/download/Police%20Facility%20Locations%20-%204326.geojson
8,ambulance.geojson,35.9 KiB,2025-01-20 19:15:44.685794,2025-01-20 19:15:44.714709,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/ambulance-station-locations/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/ec914e83-3382-4773-bbb5-634329ac3cac/resource/20c36d60-22c6-492c-a02a-9e38ed644375/download/ambulance-station-locations%20-%204326.geojson
9,fire_hydrants.geojson,19.5 MiB,2025-01-20 19:15:44.957283,2025-01-20 19:15:46.403691,City of Toronto Open Data Portal,https://open.toronto.ca/dataset/fire-hydrants/,https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/fbfcf7fa-1026-4b29-bff6-0765ca3f8a54/resource/cd55f67c-31c0-4400-8abd-fb239e55a119/download/Fire%20Hydrants%20Data%20-%204326.geojson
